In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# 0. 파일 경로
# =========================
PRICE_FILE = "3년수정주가.xlsx"
ROE_FILE = "3년분기ROE.xlsx"
DEBT_FILE = "3년분기부채비율.xlsx"
OP_FILE = "3년분기영업이익.xlsx"
KOSPI_FILE = "3년코스피지수.xlsx"

# =========================
# 1. 전략 파라미터
# =========================
N_HOLDINGS = 15
REBALANCE_EVERY = 10   # 2주 ≒ 10거래일

ROE_THRESHOLD = 20
DEBT_THRESHOLD = 200

MOM_SHORT_WINDOW = 10
MOM_LONG_WINDOW = 60

TRANSACTION_COST = 0.0
FUND_LAG_DAYS = 0
USE_NEXT_DAY_EXECUTION = False

START_DATE = None
END_DATE = None

# 비정상 가격 데이터 안전장치
MAX_DAILY_RETURN = 1.0
MIN_DAILY_RETURN = -0.95

# =========================
# 2. 데이터 로더
# =========================
def load_price_data(path):
    df = pd.read_excel(path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df.set_index("Date").sort_index()
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

def load_quarter_data_simple(path):
    df = pd.read_excel(path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df.set_index("Date").sort_index()
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

def load_kospi(path):
    df = pd.read_excel(path)
    df = df.rename(columns={df.columns[0]: "Date", df.columns[1]: "KOSPI"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df.set_index("Date").sort_index()
    df["KOSPI"] = pd.to_numeric(df["KOSPI"], errors="coerce")
    return df

def expand_quarter_to_daily(q_df, daily_index, lag_days=0):
    daily = q_df.reindex(daily_index).ffill()
    if lag_days > 0:
        daily = daily.shift(lag_days)
    return daily

# =========================
# 3. 데이터 불러오기
# =========================
price = load_price_data(PRICE_FILE)
roe_q = load_quarter_data_simple(ROE_FILE)
debt_q = load_quarter_data_simple(DEBT_FILE)
op_q = load_quarter_data_simple(OP_FILE)
kospi = load_kospi(KOSPI_FILE)

# 날짜 범위 제한
if START_DATE is not None:
    start_date = pd.to_datetime(START_DATE)
    price = price.loc[price.index >= start_date]
    kospi = kospi.loc[kospi.index >= start_date]

if END_DATE is not None:
    end_date = pd.to_datetime(END_DATE)
    price = price.loc[price.index <= end_date]
    kospi = kospi.loc[kospi.index <= end_date]

# 공통 종목만 사용
common_cols = sorted(list(set(price.columns) & set(roe_q.columns) & set(debt_q.columns) & set(op_q.columns)))
price = price[common_cols]
roe_q = roe_q[common_cols]
debt_q = debt_q[common_cols]
op_q = op_q[common_cols]

# =========================
# 4. 분기 데이터 일별 확장
# =========================
roe_daily = expand_quarter_to_daily(roe_q, price.index, FUND_LAG_DAYS)
debt_daily = expand_quarter_to_daily(debt_q, price.index, FUND_LAG_DAYS)

op_positive_q = op_q > 0
op_2q_positive_q = op_positive_q & op_positive_q.shift(1)
op_2q_positive_daily = expand_quarter_to_daily(
    op_2q_positive_q.astype(float),
    price.index,
    FUND_LAG_DAYS
) > 0

# =========================
# 5. 퀄리티 + 모멘텀 조건
# =========================
quality_filter = (
    (roe_daily > ROE_THRESHOLD) &
    (debt_daily < DEBT_THRESHOLD) &
    (op_2q_positive_daily)
)

mom_short = price / price.shift(MOM_SHORT_WINDOW) - 1
mom_long = price / price.shift(MOM_LONG_WINDOW) - 1

# =========================
# 6. 리밸런싱 날짜
# =========================
lookback_start = max(MOM_SHORT_WINDOW, MOM_LONG_WINDOW)
all_dates = price.index
signal_dates = all_dates[lookback_start::REBALANCE_EVERY]

if USE_NEXT_DAY_EXECUTION:
    trade_dates = []
    valid_signal_dates = []
    for d in signal_dates:
        loc = all_dates.get_loc(d)
        if loc + 1 < len(all_dates):
            valid_signal_dates.append(d)
            trade_dates.append(all_dates[loc + 1])
    signal_dates = pd.DatetimeIndex(valid_signal_dates)
    trade_dates = pd.DatetimeIndex(trade_dates)
else:
    trade_dates = signal_dates.copy()

# =========================
# 7. 목표 비중 생성
# =========================
target_weights = pd.DataFrame(0.0, index=price.index, columns=price.columns)

for signal_d, trade_d in zip(signal_dates, trade_dates):
    q = quality_filter.loc[signal_d]
    ms = mom_short.loc[signal_d]
    ml = mom_long.loc[signal_d]

    final_filter = q & (ms > 0) & (ml > 0)
    candidates = ms[final_filter].dropna()

    if len(candidates) == 0:
        continue

    top = candidates.sort_values(ascending=False).head(N_HOLDINGS)
    selected = top.index.tolist()

    target_weights.loc[trade_d, selected] = 1.0 / len(selected)

# 리밸런싱 사이 비중 유지
weights = target_weights.copy()
weights = weights.mask(weights.sum(axis=1) == 0)
weights = weights.ffill().fillna(0)

# =========================
# 8. 수익률 계산
# =========================
daily_ret = price.pct_change()
daily_ret = daily_ret.clip(lower=MIN_DAILY_RETURN, upper=MAX_DAILY_RETURN)
daily_ret = daily_ret.fillna(0)

portfolio_ret = (weights.shift(1).fillna(0) * daily_ret).sum(axis=1)

turnover = weights.fillna(0).diff().abs().sum(axis=1)
if len(turnover) > 0:
    turnover.iloc[0] = weights.iloc[0].abs().sum()

portfolio_ret_after_cost = portfolio_ret - turnover * TRANSACTION_COST
portfolio_cum = (1 + portfolio_ret_after_cost).cumprod()

# =========================
# 9. 벤치마크
# =========================
kospi = kospi.reindex(price.index).ffill()
kospi_ret = kospi["KOSPI"].pct_change().fillna(0)
kospi_cum = (1 + kospi_ret).cumprod()

# =========================
# 10. 성과지표 계산
# =========================
def calc_perf_stats(ret, cum):
    total_ret = cum.iloc[-1] - 1
    cagr = np.nan if cum.iloc[-1] <= 0 else cum.iloc[-1] ** (252 / len(cum)) - 1
    ann_vol = ret.std() * np.sqrt(252)
    sharpe = np.nan if (ann_vol == 0 or pd.isna(cagr)) else cagr / ann_vol

    running_max = cum.cummax()
    drawdown = cum / running_max - 1
    mdd = drawdown.min()

    return {
        "총수익률": total_ret,
        "CAGR": cagr,
        "연환산 변동성": ann_vol,
        "샤프지수": sharpe,
        "MDD": mdd
    }

port_stats = calc_perf_stats(portfolio_ret_after_cost, portfolio_cum)
kospi_stats = calc_perf_stats(kospi_ret, kospi_cum)

def format_stats_paragraph(title, stats):
    cagr_text = f"{stats['CAGR']:.2%}" if pd.notna(stats["CAGR"]) else "NaN"
    sharpe_text = f"{stats['샤프지수']:.2f}" if pd.notna(stats["샤프지수"]) else "NaN"

    return (
        f"{title}\n"
        f"총수익률: {stats['총수익률']:.2%}\n"
        f"CAGR: {cagr_text}\n"
        f"연환산 변동성: {stats['연환산 변동성']:.2%}\n"
        f"샤프지수: {sharpe_text}\n"
        f"MDD: {stats['MDD']:.2%}"
    )

# =========================
# 11. 그래프
# =========================
plt.figure(figsize=(14, 7))
plt.plot(portfolio_cum.index, portfolio_cum.values, label="Portfolio")
plt.plot(kospi_cum.index, kospi_cum.values, label="KOSPI")
plt.title("Portfolio vs KOSPI Cumulative Return")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# =========================
# 12. 결과 출력
# =========================
print("\n" + "=" * 40)
print(format_stats_paragraph("전략 포트폴리오", port_stats))
print("=" * 40)
print(format_stats_paragraph("KOSPI", kospi_stats))
print("=" * 40)